In [ ]:
# # S4 신생·쇠퇴의 재정의: 스텝별 지침
# 계획서 4.1~4.7을 기준으로, 이미 동결·실행된 것과 남은 것을 구분해서 적습니다. 모든 수치는 동결 파일(`S4_0_PREREG.json`, 해시 `44a9e523c565`)의 값이며, 결과를 보고 바꾸지 않습니다.

# ## 전체 논리
# 기존 "신생/쇠퇴"는 토픽 비중의 단순 증감이었습니다. S4는 이것을 세 축으로 다시 정의합니다.
# 1. **양:** 얼마나 변했는가
# 2. **질:** 변한 토픽이 영향력 있는 논문으로 이뤄졌는가
# 3. **구성:** 그 변화가 저널 안에서 일어났는가, 저널 비중이 바뀐 탓인가
# 세 축을 합쳐 4분면 라벨을 만들고, 그 라벨이 저널 구성을 통제해도 유지되는지(≥0.70)가 통과 기준입니다.

# ## 4.1 기간별 키워드 변화 (이월, 미실행)
# - **하는 일:** S3 L1의 구별 단어 점수를 기간마다 적용해 인접 기간 top-10 Jaccard, 탄생 단어(이전 기간 top-50에 없음), 소멸 단어를 구합니다.
# - **핵심 주의:** L1 산출물은 top-10만 저장했으므로 top-50은 새로 계산해야 합니다(`s4_1_keywords.py`).
# - **해석:** Jaccard가 낮은 토픽은 이름은 같아도 내용이 바뀐 토픽입니다. unigram과 bigram이 섞여 있어 Jaccard가 왜곡될 수 있습니다.
# - **쓰임:** 4분면 결과의 "의미 해석" 보조 증거입니다. 통과 기준은 없습니다.

# ## 4.2 양 (실행 완료)
# - **정의:** 주 대비는 2005–09 대 2020–25(`P0509`→`P2025`)입니다. 민감도 대비는 2015–19 대 2020–24입니다.
# - **두 지표:** 절대 비중 차이 Δs와 평활 로그 비율 `log((s1+0.5/N1)/(s0+0.5/N0))`를 씁니다. 임의의 1e-4는 폐기했습니다.
# - **big_change:** |Δs|가 149개 분포의 상위 3분위(경계 0.00789)이면 큽니다. 결과는 big 50개(쇠퇴 25, 성장 25)입니다.
# - **신생 정의 4종(범위로 보고):**
# | 정의 | 내용 |
# |---|---|
# | D1 | F6 원본 emerging 열 |
# | D2 | 주 대비 평활 로그 비율 상위 14 |
# | D3 | 주 대비 Δs 상위 14 |
# | D4 | 민감도 대비 Δs 상위 14 |
# - **결과:** Jaccard 0.04~0.47, 4종 모두 해당은 1개뿐입니다. **단일 "신생 토픽 목록"으로 주장하지 말고 범위로 쓰세요.**
# - **미확인:** D1이 다른 이유(F6 원본의 대비 구간)는 추정 단계입니다. F6 생성 코드를 확인해야 합니다.

# ## 4.3 질 (실행 완료)
# - **정의:** 연도×저널 셀 안의 평균 순위 백분위 `(rank−0.5)/n`입니다. 2023년 이후 논문은 제외합니다(인용 누적 부족).
# - **토픽 지표:** 평균 백분위, 상위 10% 비율, 논문 부트스트랩(B=1000) 95% 구간입니다.
# - **quality_high:** 평균 > 0.50이고 구간 하한 > 0.50일 때입니다.
# - **주의 1:** 토픽 10(0.503), 87(0.5375)처럼 0.5 근처에서 라벨이 불안정합니다. 규칙이 동결이므로 바꾸지 말고 "질 라벨은 0.5 근처에서 불안정"이라고 쓰세요.
# - **주의 2(순환):** 토픽이 SPECTER2 임베딩에서 나왔으므로 인용으로 토픽을 예측하는 모형은 쓰지 않습니다.
# - **이월:** 출판 후 3년 고정 창(OpenAlex `counts_by_year`)은 제공 연도 범위를 확인한 뒤 별도 민감도로 합니다.

# ## 4.4 4분면 (실행 완료)
# 규칙은 `big_change × 부호 × quality_high`입니다. 라벨은 5종입니다.
# - growth_highQ, growth_notHighQ, decline_highQ, decline_notHighQ, small_change
# 결과는 small_change 99, decline_notHighQ 23, growth_highQ 18, growth_notHighQ 7, decline_highQ 2입니다. 전체 149개와 A등급 112개를 **항상 병행 보고**합니다.

# ## 4.5 저널 구성 정확 분해 (실행 완료)
# - **식:** Griliches-Regev 중점 가중 분해를 씁니다.
#   `ΔS_k = Σ_j w̄_j·Δs_jk (저널 내) + Σ_j s̄_jk·Δw_j (저널 간)`
# - **불변식 확인:** 내+간=총변화(허용오차 1e-9). 실패하면 이후 수치를 쓰지 않습니다. 이번에는 통과했습니다.
# - **안정성 검정(통과 기준):** 저널 내 성분만으로 같은 규칙을 적용했을 때 big_change 토픽의 라벨이 유지되는 비율이 0.70 이상이어야 합니다. 결과는 전체 0.700, A등급 0.690입니다.
# - **해석 원칙:**
#   - 0.700은 **경계 통과**입니다. big 50개 중 35개 유지이므로 토픽 1~2개로 뒤집힙니다. "A등급만으로는 0.69"를 함께 쓰세요.
#   - 쇠퇴 big의 저널 간 성분 중앙값은 0.50(구성·내재 반반), 성장은 0.25(저널 내 지배)입니다. "쇠퇴는 전부 구성 효과"도 "전부 내재"도 기각됩니다.
#   - 총변화와 반대 부호인 토픽이 27개입니다. 구성 변화가 내재 변화를 상쇄하는 경우입니다.

# ## 4.6 회전 지수와 보조 시계열 (이월, 코드 제시 후 미실행)
# - **내용:** 토픽×저널×기간 히트맵 입력, 인접 연도 JS 거리, 주제 유효수, HHI, 기간 회전 지수(JS, Spearman, top-20 진입·퇴출).
# - **핵심:** 같은 해를 무작위 반분한 **잡음 바닥**을 함께 저장하고 `JS − 바닥`으로 보정합니다. 연 3,000편 안팎에 토픽이 149개라 표집 잡음만으로도 거리가 0보다 큽니다.
# - **세 표본:** all, fixed(고정 14개), within(저널 가중을 고정하고 저널 내 변화만).
# - **볼 것:** 2006~2008, 2020, 2024~2025의 보정 JS가 튀는지, `within` 상위 연도와 `all` 상위 연도가 같은지.
# - **쓰임:** S5 탐지의 입력입니다. 이 출력을 본 뒤 S5를 동결하므로 "완전 비열람이 아님"을 CHANGELOG에 적습니다.

# ## 4.7 정의 의존성: 지속 저자 따라감 비율 (S6 직전)
# - 신생 정의 4종 각각에서 지속 저자가 신생 토픽으로 얼마나 이동했는지를 구해 **단일 숫자 대신 범위**로 보고합니다.
# - 저자 층 변수가 필요하고 모집단이 임베딩 보유 논문의 저자이므로 S6 직전에 합니다.
# ## 결측 보정 민감도 (동결 정식 항목, 실행 완료)
# - 모형: `P(has_topic=1 | 연도, 저널, 기간×저널, log n_authors, is_solo)`, 안정화 역확률 가중(1·99 백분위 절단).
# - F2 파생 변수는 공변량에 쓰지 않습니다(F2가 임베딩 보유 논문 위에서 구축되어 순환).
# - 결과: AUC 0.733, 유효 표본 74,830/76,984, 라벨 유지 0.980(A등급 0.976), 바뀐 토픽은 26번 하나입니다.
# - **한계:** 저널×연도로 설명되는 선택만 보정합니다. 가중치 범위가 0.86~1.94로 좁다는 것은 공변량이 결측을 약하게만 설명한다는 신호이기도 합니다.
# ## 지켜야 할 규칙
# 1. 동결 판정(big 경계, 질 규칙, 안정성 0.70)은 결과를 보고 바꾸지 않습니다. 사후 관찰은 CHANGELOG에 "판정 변경 없음"으로 기록합니다.
# 2. 모든 요약은 전체 149개와 A등급 112개를 병행합니다.
# 3. 해석은 "함께 움직인다"까지만 쓰고 인과로 쓰지 않습니다.
# 4. 불변식(질량 보존, 내+간=총변화)이 깨지면 그 실행의 수치는 폐기합니다.
# ## 논문에 쓸 문장 방향
# - 주제 비중 변화 big 50개 중 쇠퇴 25개, 성장 25개이며, 쇠퇴의 절반(중앙값)은 저널 구성, 절반은 저널 내 변화로 나뉘고 성장은 저널 내 변화가 지배적이다.
# - 4분면 라벨은 저널 내 성분으로 재분류하면 70%(A등급 69%)만 유지되어 저널 구성에 민감한 편이다.
# - 신생 토픽 목록은 정의 4종 간 일치도가 낮아 단일 목록 대신 범위로 보고한다.

In [4]:
# -*- coding: utf-8 -*-
# s4_6_rotation.py: 4.6 회전 지수 + S5 입력 시계열. OpenAlex 호출 없음. 분석 집합 76,984편.
import json, warnings
from pathlib import Path
import numpy as np, pandas as pd
from scipy.spatial.distance import jensenshannon
from scipy.stats import spearmanr
warnings.filterwarnings("ignore"); pd.set_option("display.width", 200)
ROOT = Path.home() / "Research/Minnesota_Transport"
DATA, S0, S3, S4 = ROOT / "analysis_data/final", ROOT / "paper1/00_S0", ROOT / "paper1/03_S3", ROOT / "paper1/04_S4"
tf = lambda s: s.astype(str).str.strip().str.lower().isin(["1", "1.0", "true", "t", "yes"])
spec = json.loads((S4 / "S4_0_PREREG.json").read_text(encoding="utf-8")); seed = int(spec["seed"]); rng = np.random.default_rng(seed)
W = pd.read_csv(DATA / "F1_papers.csv", low_memory=False); W["pid"] = W.pid.astype(str)
EXC = set(pd.read_csv(S0 / "EXCLUDE_PIDS_FROZEN.csv", dtype=str, encoding="utf-8-sig").pid)
W = W[~W.pid.isin(EXC) & tf(W.in_sample) & W.year.between(2000, 2025) & tf(W.has_topic) & W.topic150.notna()].copy()
assert len(W) == 76984, len(W)
PRE = json.loads((ROOT / "paper1/00_S0/PREREG.json").read_text(encoding="utf-8"))
def _find(o):
    if isinstance(o, dict):
        for k, v in o.items():
            if k == "fixed_venues": return v
            r = _find(v)
            if r is not None: return r
FIXED = set(_find(PRE)); assert len(FIXED) == 14, len(FIXED)
W["topic"] = W.topic150.astype(int); W["fixed"] = W.venue_short.isin(FIXED)
topics = np.array(sorted(pd.read_csv(S3 / "L7_topic_grade.csv").topic.astype(int))); K = len(topics); tix = {t: i for i, t in enumerate(topics)}
W["ti"] = W.topic.map(tix); years = np.arange(2000, 2026)
venues = np.array(sorted(W.venue_short.unique())); vix = {v: i for i, v in enumerate(venues)}; W["vi"] = W.venue_short.map(vix)

def vec(d): return np.bincount(d.ti.values, minlength=K).astype(float)
def norm(c): return c / c.sum() if c.sum() > 0 else c
js = lambda a, b: float(jensenshannon(a, b, base=2))
def noise_floor(d, R=50):
    x = d.ti.values; out = []
    for _ in range(R):
        p = rng.permutation(len(x)); h = len(x) // 2
        out.append(js(norm(np.bincount(x[p[:h]], minlength=K).astype(float)), norm(np.bincount(x[p[h:]], minlength=K).astype(float))))
    return float(np.median(out))

cnt = np.zeros((len(years), len(venues), K))
for (y, v), d in W.groupby(["year", "vi"]): cnt[y - 2000, v] = vec(d)
wbar = cnt.sum((0, 2)) / cnt.sum()
fixed_v = np.array([bool(W.loc[W.vi == i, "fixed"].mean() > 0.5) for i in range(len(venues))])
print(f"저널 {len(venues)}개 | 고정 저널로 판정된 수 {int(fixed_v.sum())} (14여야 함)")
D3 = set()
try:
    DS = pd.read_csv(S4 / "definition_sensitivity.csv"); D3 = set(DS[tf(DS.D3)].topic.astype(int))
except Exception as e: print("※ D3 생략:", e)
d3m = np.array([t in D3 for t in topics])

series = {}
for name in ["all", "fixed", "within"]:
    S = []
    for yi in range(len(years)):
        c = cnt[yi]
        if name == "all": v = norm(c.sum(0))
        elif name == "fixed": v = norm(c[fixed_v].sum(0))
        else:
            nj = c.sum(1); ok = nj > 0; sj = np.zeros_like(c); sj[ok] = c[ok] / nj[ok, None]
            ww = np.where(ok, wbar, 0); v = norm((ww[:, None] * sj).sum(0))
        S.append(v)
    series[name] = np.array(S)
rows = []
for name, S in series.items():
    for yi, y in enumerate(years):
        s = S[yi]; p = s[s > 0]; H = -(p * np.log(p)).sum()
        npap = int(cnt[yi].sum() if name != "fixed" else cnt[yi][fixed_v].sum())
        r = dict(sample=name, year=int(y), n_papers=npap, eff_topics=float(np.exp(H)), hhi=float((s ** 2).sum()), d3_share=float(s[d3m].sum()))
        if yi > 0:
            r["js_adj"] = js(S[yi - 1], s)
            if name != "within":
                sub = W[W.year == y] if name != "fixed" else W[(W.year == y) & W.fixed]
                sub0 = W[W.year == y - 1] if name != "fixed" else W[(W.year == y - 1) & W.fixed]
                r["noise_floor"] = float(np.mean([noise_floor(sub), noise_floor(sub0)])); r["js_adj_corrected"] = r["js_adj"] - r["noise_floor"]
            else: r["noise_floor"] = np.nan; r["js_adj_corrected"] = np.nan
        rows.append(r)
Y = pd.DataFrame(rows); Y.to_csv(S4 / "yearly_series_for_S5.csv", index=False, encoding="utf-8-sig")
for nm in ["all", "fixed", "within"]: np.save(S4 / f"yearly_topic_shares_{nm}.npy", series[nm])
np.save(S4 / "yearly_venue_topic_counts.npy", cnt)
pd.Series(venues).to_csv(S4 / "venues_order.csv", index=False, header=["venue"]); pd.Series(topics).to_csv(S4 / "topics_order.csv", index=False, header=["topic"])

print("[연도 인접 JS: 보정값 상위 5, 표본별]")
for name in ["all", "fixed"]:
    print(name); print(Y[Y["sample"] == name].dropna(subset=["js_adj_corrected"]).sort_values("js_adj_corrected", ascending=False).head(5)[["year", "n_papers", "js_adj", "noise_floor", "js_adj_corrected"]].round(4).to_string(index=False))
print("within 인접 JS 상위 5:\n", Y[Y["sample"] == "within"].dropna(subset=["js_adj"]).sort_values("js_adj", ascending=False).head(5)[["year", "js_adj"]].round(4).to_string(index=False))
print("2006~2008, 2020, 2024~2025 (all):\n", Y[(Y["sample"] == "all") & Y.year.isin([2006, 2007, 2008, 2020, 2024, 2025])][["year", "n_papers", "js_adj", "noise_floor", "js_adj_corrected", "eff_topics", "hhi"]].round(4).to_string(index=False))

per = {"P0004": (2000, 2004), "P0509": (2005, 2009), "P1014": (2010, 2014), "P1519": (2015, 2019), "P2025": (2020, 2025)}
pv = {k: norm(sum(cnt[y - 2000].sum(0) for y in range(a, b + 1))) for k, (a, b) in per.items()}; ks = list(per); rr = []
for a, b in zip(ks[:-1], ks[1:]):
    t0, t1 = set(np.argsort(-pv[a])[:20]), set(np.argsort(-pv[b])[:20])
    rr.append(dict(pair=f"{a}->{b}", js=js(pv[a], pv[b]), spearman=float(spearmanr(pv[a], pv[b])[0]), top20_entry=len(t1 - t0), top20_exit=len(t0 - t1)))
R = pd.DataFrame(rr); R.to_csv(S4 / "rotation_index_periods.csv", index=False, encoding="utf-8-sig"); print("\n[기간 회전 지수]\n", R.round(4).to_string(index=False))
H = []
for k, (a, b) in per.items():
    c = sum(cnt[y - 2000] for y in range(a, b + 1))
    for vi, v in enumerate(venues):
        for ti_, t in enumerate(topics):
            if c[vi, ti_] > 0: H.append((k, v, int(t), int(c[vi, ti_]), c[vi, ti_] / c[vi].sum()))
pd.DataFrame(H, columns=["period", "venue", "topic", "n", "share_in_venue"]).to_csv(S4 / "heatmap_topic_venue_period.csv", index=False, encoding="utf-8-sig")
with (S4 / "CHANGELOG_S4.md").open("a", encoding="utf-8") as f:
    f.write(f"- {pd.Timestamp.now().isoformat(timespec='seconds')} | 4.6 실행(이월 항목): 연도 인접 JS(잡음 바닥=같은 해 반분 중앙값 50회), 세 표본(all/fixed/within), 기간 회전 지수, 토픽x저널x기간 표. S5 입력 시계열 저장 | S4 4.6 | S5\n")
print("저장 완료")

저널 34개 | 고정 저널로 판정된 수 14 (14여야 함)
[연도 인접 JS: 보정값 상위 5, 표본별]
all
 year  n_papers  js_adj  noise_floor  js_adj_corrected
 2025      7206  0.1186       0.1230           -0.0043
 2022      5649  0.1346       0.1403           -0.0057
 2020      4794  0.1552       0.1613           -0.0061
 2006      1623  0.2383       0.2534           -0.0150
 2016      3278  0.1696       0.1851           -0.0155
fixed
 year  n_papers  js_adj  noise_floor  js_adj_corrected
 2021      4287  0.1528       0.1661           -0.0133
 2006      1593  0.2394       0.2554           -0.0161
 2022      4114  0.1437       0.1618           -0.0181
 2018      3382  0.1609       0.1853           -0.0244
 2020      3744  0.1520       0.1777           -0.0257
within 인접 JS 상위 5:
  year  js_adj
 2001  0.4345
 2002  0.3590
 2005  0.3200
 2004  0.3155
 2006  0.3116
2006~2008, 2020, 2024~2025 (all):
  year  n_papers  js_adj  noise_floor  js_adj_corrected  eff_topics    hhi
 2006      1623  0.2383       0.2534           -0.0150   

In [5]:
#

In [6]:
# -*- coding: utf-8 -*-
# s4_6b_floor_check.py: 풀 재표집 바닥으로 연도 인접 JS 재점검. 읽기 전용 탐색(동결 파일 변경 없음).
import json, warnings
from pathlib import Path
import numpy as np, pandas as pd
from scipy.spatial.distance import jensenshannon
warnings.filterwarnings("ignore"); pd.set_option("display.width", 200)
R_ = 1000          # 귀무 재표집 횟수
ROOT = Path.home() / "Research/Minnesota_Transport"
DATA, S0, S3, S4 = ROOT / "analysis_data/final", ROOT / "paper1/00_S0", ROOT / "paper1/03_S3", ROOT / "paper1/04_S4"
tf = lambda s: s.astype(str).str.strip().str.lower().isin(["1", "1.0", "true", "t", "yes"])
rng = np.random.default_rng(2026)
PRE = json.loads((S0 / "PREREG.json").read_text(encoding="utf-8"))
def _find(o):
    if isinstance(o, dict):
        for k, v in o.items():
            if k == "fixed_venues": return v
            r = _find(v)
            if r is not None: return r
FIXED = set(_find(PRE)); assert len(FIXED) == 14, len(FIXED)

W = pd.read_csv(DATA / "F1_papers.csv", low_memory=False); W["pid"] = W.pid.astype(str)
EXC = set(pd.read_csv(S0 / "EXCLUDE_PIDS_FROZEN.csv", dtype=str, encoding="utf-8-sig").pid)
W = W[~W.pid.isin(EXC) & tf(W.in_sample) & W.year.between(2000, 2025) & tf(W.has_topic) & W.topic150.notna()].copy()
print(f"분석 집합 {len(W):,}편 (76,984여야 함)"); assert len(W) == 76984
topics = np.array(sorted(pd.read_csv(S3 / "L7_topic_grade.csv").topic.astype(int))); K = len(topics)
W["ti"] = W.topic150.astype(int).map({t: i for i, t in enumerate(topics)}).values
W["fixed"] = W.venue_short.isin(FIXED); W["vi"] = W.venue_short.astype("category").cat.codes
years = np.arange(2000, 2026)

def dist(x): c = np.bincount(x, minlength=K).astype(float); return c / c.sum()
js = lambda a, b: float(jensenshannon(a, b, base=2))

def pool_null(x0, x1):
    """x0, x1: 두 해의 토픽 인덱스. 풀에서 같은 크기로 재표집한 JS 분포."""
    pool = np.concatenate([x0, x1]); n0 = len(x0); out = np.empty(R_)
    for r in range(R_):
        p = rng.permutation(len(pool)); out[r] = js(dist(pool[p[:n0]]), dist(pool[p[n0:]]))
    return out

def within_null(d0, d1):
    """저널 구성은 유지하고 저널x연도 셀 안에서만 토픽을 섞는 귀무(풀은 두 해의 같은 저널 논문)."""
    out = np.empty(R_); j0, j1 = d0.vi.values, d1.vi.values; t0, t1 = d0.ti.values, d1.ti.values
    pooled = {v: np.concatenate([t0[j0 == v], t1[j1 == v]]) for v in np.union1d(j0, j1)}
    n0 = {v: int((j0 == v).sum()) for v in pooled}
    for r in range(R_):
        a, b = [], []
        for v, arr in pooled.items():
            p = rng.permutation(len(arr)); a.append(arr[p[:n0[v]]]); b.append(arr[p[n0[v]:]])
        out[r] = js(dist(np.concatenate(a)), dist(np.concatenate(b)))
    return out

rows = []
for name in ["all", "fixed", "within"]:
    base = W if name != "fixed" else W[W.fixed]
    for y in years[1:]:
        d0, d1 = base[base.year == y - 1], base[base.year == y]
        if len(d0) < 20 or len(d1) < 20: continue
        obs = js(dist(d0.ti.values), dist(d1.ti.values))
        null = within_null(d0, d1) if name == "within" else pool_null(d0.ti.values, d1.ti.values)
        rows.append(dict(sample=name, year=int(y), n0=len(d0), n1=len(d1), js_obs=obs, floor_med=float(np.median(null)),
                         floor_p95=float(np.percentile(null, 95)), corrected=obs - float(np.median(null)), p_emp=float((null >= obs).mean())))
    print(f"{name} 완료")
Y = pd.DataFrame(rows); Y.to_csv(S4 / "yearly_js_poolfloor_check.csv", index=False, encoding="utf-8-sig")
for name in ["all", "fixed", "within"]:
    s = Y[Y["sample"] == name]
    print(f"\n[{name}] 보정값>0 연도 {int((s.corrected > 0).sum())}/{len(s)} | p<0.05 연도 {int((s.p_emp < 0.05).sum())}")
    print(s.sort_values("corrected", ascending=False).head(6)[["year", "n0", "n1", "js_obs", "floor_med", "corrected", "p_emp"]].round(4).to_string(index=False))
print("\n[2006~2008, 2019~2021, 2024~2025 (all)]")
print(Y[(Y["sample"] == "all") & Y.year.isin([2006, 2007, 2008, 2019, 2020, 2021, 2024, 2025])][["year", "n0", "n1", "js_obs", "floor_med", "corrected", "p_emp"]].round(4).to_string(index=False))
with (S4 / "CHANGELOG_S4.md").open("a", encoding="utf-8") as f:
    f.write(f"- {pd.Timestamp.now().isoformat(timespec='seconds')} | 4.6b 바닥 재점검(탐색, 동결 판정 없음): 두 해 풀에서 같은 크기 재표집 {R_}회(within은 저널x연도 셀 내 토픽 섞기). 반분 바닥의 과대평가 점검 | S4 4.6 | S5 바닥 규칙\n")
print("\n저장: yearly_js_poolfloor_check.csv")

분석 집합 76,984편 (76,984여야 함)
all 완료
fixed 완료
within 완료

[all] 보정값>0 연도 22/25 | p<0.05 연도 15
 year   n0   n1  js_obs  floor_med  corrected  p_emp
 2006 1515 1623  0.2383     0.1811     0.0572    0.0
 2020 3866 4794  0.1552     0.1121     0.0431    0.0
 2016 3029 3278  0.1696     0.1316     0.0380    0.0
 2022 5440 5649  0.1346     0.0982     0.0364    0.0
 2025 7167 7206  0.1186     0.0863     0.0323    0.0
 2018 3364 3910  0.1541     0.1231     0.0310    0.0

[fixed] 보정값>0 연도 21/25 | p<0.05 연도 13
 year   n0   n1  js_obs  floor_med  corrected  p_emp
 2006 1499 1593  0.2394     0.1819     0.0575    0.0
 2021 3744 4287  0.1528     0.1168     0.0360    0.0
 2022 4287 4114  0.1437     0.1129     0.0308    0.0
 2018 2936 3382  0.1609     0.1309     0.0300    0.0
 2016 2767 2875  0.1659     0.1373     0.0286    0.0
 2020 3229 3744  0.1520     0.1245     0.0275    0.0

[within] 보정값>0 연도 21/25 | p<0.05 연도 12
 year   n0   n1  js_obs  floor_med  corrected  p_emp
 2006 1515 1623  0.2383     0.1828  

In [7]:
#

In [ ]:
# # 4.6b 해석: 이전 "연 단위 신호 없음"은 바닥 정의 탓이었습니다

# ## 결론

# **풀 재표집 바닥으로 바꾸자 보정값이 대부분 양수로 바뀌었습니다.** `all`에서 22/25 연도가 양수이고 15개 연도가 p<0.05입니다. 이전의 "전부 음수"는 반분 바닥이 인접 연도 비교의 잡음을 과대평가한 결과였고(제가 추정했던 방향과 일치), 연 단위 탐지가 가능합니다. 다만 아래 주의가 있습니다.

# ## 핵심 관찰

# | 관찰 | 의미 |
# |---|---|
# | `all` 22/25 양수, p<0.05 15개 | 연 단위 신호 있음. 세 표본(all 15, fixed 13, within 12)이 비슷해 저널 구성만의 효과는 아님 |
# | **2006 보정 +0.057(최대, p=0)** | 세 표본 모두 1위. 2005→2006 구조 변화 후보. 다만 2006 n=1,623, 2005 n=1,515로 표본이 작은 구간 |
# | **2007 +0.009, p=0.212** | 유일하게 잡음과 구분 안 되는 구간. 2007년 n=1,271로 소표본이고 임베딩 보유율이 꺾인 해와 겹침. 아티팩트 후보 표시와 일치 |
# | 2008 +0.027(p=0.009) | 2007 이후 회복. 2007이 "꺾임"이라면 2007-08 사이 양방향 점프일 수 있음(추정) |
# | **2020 +0.043, 2021 +0.026** | 2020년 논문 수 급증(3,866→4,794) 구간. 구성 변화인지 표본 증가에 따른 아티팩트인지는 이 결과만으로 구분 못 함 |
# | 2025 +0.032 | 부분연도·보유율 하락 후보 연도와 겹침 |

# ## 주의 (과해석 방지)

# 1. **바닥은 논문 독립을 가정합니다.** 같은 저널·공저·후속 논문의 상관이 있으면 실제 잡음은 더 크므로 p는 낙관적입니다. "15개 연도 유의"를 그대로 믿지 마세요.
# 2. **보정값 크기가 표본 크기에 영향받습니다.** 2006이 1위인 것도 소표본 효과가 남았을 수 있습니다. 보정값은 순위용이지 크기 비교용이 아닙니다.
# 3. **연 단위 JS는 "이웃 연도와 다르다"만 말합니다.** 체제 전환은 "구간 전체가 다른 수준으로 이동"이므로, 이 결과가 곧 체제 변화점은 아닙니다. 그건 S5 제약 군집화가 판단합니다.
# 4. `within`에서도 2006, 2016, 2020이 양수라 저널 구성만으로는 설명되지 않습니다. 단, `within` 귀무는 다른 정의이므로 `all`과 숫자를 직접 비교하지 마세요.
# 5. 이 점검은 탐색이고 동결 판정이 아닙니다.

# ## S5-0에 반영할 것

# 1. 고정 저널은 **PREREG 14개 목록**(`venue_short.isin(FIXED)`).
# 2. **JPT 2019**를 저널 단위 아티팩트 후보에 추가(T-IV 2025, T-ITS 2008과 같은 부류).
# 3. **바닥 규칙:** 연도 인접 비교의 귀무는 풀 재표집(같은 크기)으로 사전등록. 반분 바닥은 폐기.
# 4. **2007 표시 유지:** 4.6b에서 유일하게 비유의였으므로 "측정 아티팩트 후보" 근거로 CHANGELOG에 적습니다.
# 5. 탐지 단위는 **연 벡터 사용 가능**으로 두되, 5.5의 라벨 순열·완만 추세 귀무는 그대로 유지(독립 가정 문제를 보완하는 장치).
# 6. CHANGELOG 문구에 "4.6, 4.6b 산출물은 동결 전에 열람했음"을 명시.

# ## 기록

# ```bash
# cd ~/Research/Minnesota_Transport
# echo "- 2026-10-16 | 4.6b 결과(탐색, 동결 판정 없음): 풀 재표집 바닥에서 all 보정값>0 22/25, p<0.05 15개, fixed 21/25(13), within 21/25(12). 이전 반분 바닥의 전부 음수는 바닥 과대평가 탓. 2006 최대(+0.057), 2007만 비유의(+0.009, p=0.212, n=1,271), 2020 +0.043, 2025 +0.032. 재표집은 논문 독립 가정이라 p는 낙관적일 수 있음 | S4 4.6b | S5-0 바닥 규칙" >> paper1/04_S4/CHANGELOG_S4.md
# ```

# ## 다음 순서

# 1. `s4_1_output.txt` 공유(아직 못 받았습니다)
# 2. `s4_close.py` 점검(`COMMIT=False`)
# 3. 위 6가지를 반영한 `s5_0_freeze.py` 수정본: `EVENTS_FROZEN*.csv` 존재와 S1.9 완료 여부를 먼저 알려 주시면 사건 부분까지 확정해서 드리겠습니다.

In [ ]:
# # S4 진행 현황 업데이트 (2026-10-16 기준)

# 첨부 문서는 앞서 드린 지침 문서 그대로라 새 정보가 없습니다. 아래는 이후 대화에서 실행·확인된 결과를 반영한 것입니다.

# ## 1. 한눈에 보기

# | 항목 | 상태 | 핵심 결과 |
# |---|---|---|
# | S4-0 동결 | 완료 | 해시 `44a9e523c565`, 커밋 `1f36d60`, 태그 `S4-0-freeze-2026-10-16` |
# | 4.2 양, 4.4 4분면, 4.5 분해, 결측 보정 | 실행 완료 | 아래 2번 |
# | 4.1 키워드 변화 | 실행 완료 | 인접 기간 top-10 Jaccard 중앙값 0.25→0.33→0.38→0.43 |
# | 4.6 회전 지수 | 실행 완료(고정 저널 14개 기준 재실행) | 기간 Spearman 0.91→0.83→0.76→**0.32** |
# | 4.6b 바닥 재점검 | 실행 완료(탐색) | 풀 재표집 바닥에서 all 보정 JS>0인 연도 22/25 |
# | D1 진단 | 실행 완료(사후) | 2005-09→2015-19 단순 차이와 가장 비슷(Jaccard 0.40) |
# | 4.7, 3년 고정 창 | 이월 | S6 직전 / counts_by_year 범위 확인 후 |
# | **커밋·태그** | **미완, 잘못된 태그 존재** | 아래 4번 |

# ## 2. 본 계산 결과 (동결 판정, 변경 없음)

# - 4분면 전체: small_change 99, decline_notHighQ 23, growth_highQ 18, growth_notHighQ 7, decline_highQ 2.
# - 안정성: 전체 **0.700**(기준 0.70, 경계 통과), A등급 0.690. big 50개 중 35개 유지라 토픽 1~2개로 뒤집힙니다.
# - 쇠퇴 big 25개의 저널 간 성분 중앙값 0.50, 성장 big 25개 0.25. 총변화와 반대 부호 토픽 27개.
# - 결측 보정: AUC 0.733, 유효 표본 74,830/76,984, 라벨 유지 0.980(A등급 0.976), 바뀐 토픽은 26번 하나.
# - 신생 정의 4종 Jaccard 0.04~0.47, 4종 모두 해당 1개. 단일 목록 대신 범위로 보고합니다.
# - 토픽 38은 이 대비에서 small_change여서 "구성 효과 전형"이라는 사전 추정은 지지되지 않았습니다.

# ## 3. 이월 항목에서 새로 확인된 것 (탐색, 판정 아님)

# - **고정 저널은 PREREG 14개가 정답**입니다. F1의 `fixed_venue` 열은 JPT를 더 포함해 15개인데, JPT는 2019년 논문이 0편이라 규칙에 걸려 탈락해야 맞습니다. F1 열이 어긋난 원인은 미확인이고, JPT 2019는 S5 아티팩트 후보에 추가합니다. S3 L4의 `nonfixed_share`는 F1 열 기준이라 약간 어긋나지만 기술 정보라 등급 영향은 없습니다.
# - **연 단위 JS:** 반분 바닥에서는 보정값이 전부 음수였는데, 이는 바닥이 인접 연도 비교의 잡음을 과대평가한 탓이었습니다. 풀 재표집 바닥에서는 all 22/25, fixed 21/25, within 21/25가 양수입니다. 2006이 최대(+0.057)이고, 2007만 잡음과 구분되지 않았습니다(+0.009, p=0.212, n=1,271). 재표집은 논문 독립을 가정하므로 p는 낙관적일 수 있습니다.
# - **기간 회전:** 마지막 쌍(2015-19→2020-25)에서만 급변합니다. 2020-25가 6년이고 논문이 많은 기간 정의 영향은 배제하지 못했습니다.
# - **키워드:** Jaccard가 최저인 토픽 133, 65, 148, 147, 55는 카드를 읽어 볼 후보이고, 147과 49는 S3 C등급이었습니다.
# - **D1:** F6 emerging은 "최근 급성장"보다 "2005-09 대비 2015-19 성장"에 가깝다는 정황입니다(추정, F6 생성 코드 확인 전). 이것이 D1이 D2~D4와 다른 가장 유력한 설명입니다.

# ## 4. 문제: 커밋이 실제로는 되지 않았습니다

# - 마지막 로그에서 `git add`가 없는 경로(`paper1/s4_main.py`)로 통째로 실패했고, 커밋 없이 태그만 생성됐습니다. `S4-main-2026-10-16`은 그 전에 이미 있던 태그이고, `S4-final-2026-10-16`은 제 코드가 add 실패를 검사하지 않아 빈 상태로 만들어졌습니다. 둘 다 잘못된 커밋을 가리킵니다.
# - 04_S4 산출물 전부가 아직 미추적(`??`)입니다. `s4_main.py`, `s4_6_rotation.py`, `s4_6b_floor_check.py`는 `paper1/`에 파일로 저장된 흔적이 없습니다. 재현 코드가 없는 상태일 수 있습니다.
# - S3 마무리 산출물(`S3_MANIFEST.json`, `topic_evidence_cards.csv`, `supplement_label_grid.pdf` 등)도 미추적이어서, `S3-close` 커밋 여부를 확인해야 합니다.
# - `git status`의 `modified: paper1`은 서브모듈이나 중첩 저장소 신호일 수 있습니다(추정).

# ## 5. 앞으로 할 일

# 1. **진단(읽기 전용):** `git log --oneline -5`, 두 태그의 `git show --stat`, `ls paper1/s4_*.py`, `git ls-files --stage paper1`, `git tag -l "S3-*"` 출력 확인.
# 2. **S4 코드 저장:** `s4_main.py`, `s4_6_rotation.py`, `s4_6b_floor_check.py`를 에디터로 저장. 없으면 재현 불가이므로 가장 먼저 합니다.
# 3. **태그 정리·재커밋:** 잘못된 태그 두 개를 로컬에서 삭제하고, 존재하는 파일만 add한 뒤 add 성공을 확인하고 커밋·태그합니다. 본 계산과 이월이 한 커밋에 섞이므로 `S4-main`은 만들지 않고 CHANGELOG에 정정을 기록합니다.
# 4. **S3 커밋 상태 확인** 후 필요하면 같은 방식으로 정리합니다.
# 5. **S5-0 동결 전 결정:** `event_year_rule` 추가 여부(큰 사건은 `E_share5`, 진입 사건은 `E1_effective_entry`), AMAR 확인 또는 `ACK_PENDING=True`. 위 1~4가 끝나기 전에는 동결하지 마세요.
# 6. 이월: 토픽 0, 147, 88, 45, 133 카드 해석 메모, F6 생성 코드 확인(D1), 4.7(S6 직전), 3년 고정 창.

# ## 6. 논문에 쓸 한계 (S4 관련 누적)

# - 안정성은 경계 통과(0.700, A등급 0.690)이고 질 라벨은 0.5 근처에서 불안정합니다.
# - 신생 토픽은 정의 의존적이라 범위로 보고합니다.
# - 분석 집합은 임베딩 보유 논문 83.9%이며, 결측 보정은 저널×연도로 설명되는 선택만 고칩니다.
# - 인용은 단일 스냅샷입니다.
# - 고정 저널은 PREREG 14개를 쓰고, F1 열과의 불일치 원인은 미확인입니다.
# - 4.6, 4.6b 결과를 본 뒤 S5를 동결하므로 완전 비열람이 아닙니다. 4.6b 재표집은 논문 독립을 가정합니다.

# 1번 진단 출력을 보내 주시면 정리 방법을 확정하겠습니다.